In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import datasets, transforms
import torchvision.utils as vutils

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
FOOD_DATASET_ROOT = os.path.join(script_dir, "Files", "Image Classification")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available() 
    else "cpu"
)

BATCH_SIZE = 128
IMAGE_SIZE = 64
CHANNELS_IMG = 3        
Z_DIM = 100            
FEATURES_generator = 64       
FEATURES_discriminator = 64      
NUM_EPOCHS = 45     
LEARNING_RATE = 2e-4
BETA1 = 0.5             

class Generator(nn.Module):
    def __init__(self, z_dim, channels_img, features_g):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, features_g * 16, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(features_g * 16),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 16, features_g * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 8, features_g * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 4, features_g * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 2, channels_img, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh() 
        )

    def forward(self, x):
        return self.net(x)

class Discriminator(nn.Module):
    def __init__(self, channels_img, features_d):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels_img, features_d, kernel_size=4, stride=2, padding=1, bias=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d, features_d * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 2, features_d * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 4, features_d * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 8, 1, kernel_size=4, stride=1, padding=0, bias=True)
        )

    def forward(self, x):
        return self.net(x)

def initialize_weights(model):
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0)

def main():
    transforms_pipeline = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.5] * CHANNELS_IMG, [0.5] * CHANNELS_IMG),
    ])

    train_path = os.path.join(FOOD_DATASET_ROOT, "Train")
    test_path = os.path.join(FOOD_DATASET_ROOT, "Test")

    if not os.path.exists(train_path) or not os.path.exists(test_path):
        raise FileNotFoundError(
            f"Failed to Find your Train & Test Data")

    train_dataset = datasets.ImageFolder(root=train_path, transform=transforms_pipeline)
    test_dataset = datasets.ImageFolder(root=test_path, transform=transforms_pipeline)
    combined_dataset = ConcatDataset([train_dataset, test_dataset])

    dataloader = DataLoader(
        combined_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=4, 
        pin_memory=True if DEVICE.type == "cuda" else False
    )

    generator = Generator(Z_DIM, CHANNELS_IMG, FEATURES_generator).to(DEVICE)
    discriminator = Discriminator(CHANNELS_IMG, FEATURES_discriminator).to(DEVICE)
    initialize_weights(generator)
    initialize_weights(discriminator)

    opt_generator = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
    opt_discriminator = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
    criterion = nn.BCEWithLogitsLoss()
    
    scaler = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
    fixed_noise = torch.randn(32, Z_DIM, 1, 1, device=DEVICE)

    os.makedirs("food_gan_outputs", exist_ok=True)
    print(f"Dataset compiled! Combined total of {len(combined_dataset)} food images detected.")
    print(f"Detected subfolders automatically: {train_dataset.classes}")
    print(f"Starting training loop on device: {DEVICE}...\n")

    for epoch in range(NUM_EPOCHS):
        for batch_idx, (real_imgs, _) in enumerate(dataloader):
            real_imgs = real_imgs.to(DEVICE)
            batch_size = real_imgs.size(0)
            
            real_labels = torch.full((batch_size, 1, 1, 1), 0.9, device=DEVICE)
            fake_labels = torch.zeros((batch_size, 1, 1, 1), device=DEVICE)
            
            opt_discriminator.zero_grad(set_to_none=True)
            
            with torch.cuda.amp.autocast(enabled=(scaler is not None)):
                output_real = discriminator(real_imgs)
                loss_discriminator_real = criterion(output_real, real_labels)
                
                noise = torch.randn(batch_size, Z_DIM, 1, 1, device=DEVICE)
                fake_imgs = generator(noise)
                output_fake = discriminator(fake_imgs.detach())
                loss_discriminator_fake = criterion(output_fake, fake_labels)
                
                loss_discriminator = loss_discriminator_real + loss_discriminator_fake
                
            if scaler:
                scaler.scale(loss_discriminator).backward()
                scaler.step(opt_discriminator)
            else:
                loss_discriminator.backward()
                opt_discriminator.step()

            opt_generator.zero_grad(set_to_none=True)
            
            with torch.cuda.amp.autocast(enabled=(scaler is not None)):
                output_fake_for_generator = discriminator(fake_imgs)
                loss_generator = criterion(output_fake_for_generator, torch.ones_like(real_labels))
                
            if scaler:
                scaler.scale(loss_generator).backward()
                scaler.step(opt_generator)
                scaler.update()
            else:
                loss_generator.backward()
                opt_generator.step()

            if batch_idx % 10 == 0:
                print(
                    f"Epoch {epoch+1} | Batch {batch_idx} "
                    f"Loss Discriminator : {loss_discriminator:.4f}, Loss Generator : {loss_generator:.4f}"
                )
                
        with torch.no_grad():
            fake_snapshot = generator(fixed_noise)
            grid = vutils.make_grid(fake_snapshot, normalize=True)
            vutils.save_image(grid, f"food_gan_outputs/epoch_{epoch+1}.png")
            print(f"Snapshot Saved : food_gan_outputs/epoch_{epoch+1}.png")

if __name__ == "__main__":
    main()